# Test Kamera

In [13]:
import cv2

cap = cv2.VideoCapture(0)

print("Kamera terbuka!")
print("Tekan tombol 'q' untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Gagal membaca dari kamera.")
        break
    
    # Menampilkan Frame (Capture)
    cv2.imshow('Test Webcame', frame)

    # Logic quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Kamera terbuka!
Tekan tombol 'q' untuk keluar.


# Menghitung Sudut

In [15]:
import math
from typing import Tuple

def calculate_angle(a: Tuple[float, float], b: Tuple[float, float], c: Tuple[float, float]) -> float:
    """Menghitung sudut (derajat) di titik B dari tiga titik koordinat A, B, C."""
    # Vektor BA
    ba_x = a[0] - b[0]
    ba_y = a[1] - b[1]
    
    # Vektor BC
    bc_x = c[0] - b[0]
    bc_y = c[1] - b[1]
    
    # Dot product dan Magnitudo (panjang vektor)
    dot_product = ba_x * bc_x + ba_y * bc_y
    mag_ba = math.sqrt(ba_x**2 + ba_y**2)
    mag_bc = math.sqrt(bc_x**2 + bc_y**2)
    
    # Menghindari pembagian dengan nol
    if mag_ba == 0 or mag_bc == 0:
        return 0.0
        
    # Hitung cosinus sudut
    cosine_angle = dot_product / (mag_ba * mag_bc)
    
    # Batasi nilai -1.0 s.d 1.0 agar fungsi acos tidak error karena presisi float
    cosine_angle = max(-1.0, min(1.0, cosine_angle))
    
    # Ubah radian ke derajat
    angle = math.degrees(math.acos(cosine_angle))
    return angle


Menggambar Titik Tubuh

In [19]:
import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# --- PARAMETER HASIL EKSPERIMEN SABRINA ---
THRESHOLD_ANGLE = 70.0       # Batas mulai bungkuk
ALERT_FRAME_LIMIT = 30       # Harus bungkuk selama 30 frame berturut-turut

# Variabel counter
slouch_frame_count = 0

cap = cv2.VideoCapture(0)

print("Detektor postur aktif. Coba membungkuk selama 2 detik untuk menguji alarm!")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)
    
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # Ambil titik (Hidung, Bahu Kiri, Bahu Kanan)
        nose = landmarks[mp_pose.PoseLandmark.NOSE]
        left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER]
        
        A = (nose.x, nose.y)
        B = (left_shoulder.x, left_shoulder.y)
        C = (right_shoulder.x, right_shoulder.y)
        
        # Hitung sudut di Hidung (A)
        angle = calculate_angle(B, A, C)
        
        # Logika Peringatan
        if angle >= THRESHOLD_ANGLE:
            slouch_frame_count += 1
        else:
            slouch_frame_count = 0  # Reset jika kembali tegak
            
        # Tentukan status postur
        is_slouching = slouch_frame_count >= ALERT_FRAME_LIMIT
        
        # Tentukan warna garis: Merah jika bungkuk, Hijau jika tegak
        color = (0, 0, 255) if is_slouching else (0, 255, 0)
        
        # --- Visualisasi ---
        h, w, _ = frame.shape
        pixel_A = (int(A[0] * w), int(A[1] * h))
        pixel_B = (int(B[0] * w), int(B[1] * h))
        pixel_C = (int(C[0] * w), int(C[1] * h))
        
        # Gambar segitiga dengan warna dinamis
        cv2.line(frame, pixel_A, pixel_B, color, 2)
        cv2.line(frame, pixel_B, pixel_C, color, 2)
        cv2.line(frame, pixel_C, pixel_A, color, 2)
        
        # Tampilkan status teks peringatan di layar jika bungkuk
        if is_slouching:
            cv2.putText(
                frame, 
                "WARNING: BUNGKUK! TEGAKKAN BADAN!", 
                (50, 50),  # Koordinat teks di pojok kiri atas
                cv2.FONT_HERSHEY_SIMPLEX, 
                0.8, 
                (0, 0, 255), 
                2
            )
            
        # Tampilkan nilai sudut saat ini
        cv2.putText(
            frame, 
            f"Sudut: {angle:.1f} deg", 
            (pixel_A[0] - 60, pixel_A[1] - 20), # Menggunakan pixel_A
            cv2.FONT_HERSHEY_SIMPLEX, 
            0.6, 
            (255, 255, 255), 
            2
        )

        
    cv2.imshow('Detektor Postur Real-time', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()


Detektor postur aktif. Coba membungkuk selama 2 detik untuk menguji alarm!


In [25]:
from plyer import notification

# Mengirimkan notifikasi desktop sekali saja
notification.notify(
    title="Peringatan Postur!",
    message="Dudukmu bungkuk, Sabrina. Tegakkan punggungmu ya!",
    app_name="Stop Bungkuk",
    timeout=5  # Notifikasi hilang setelah 5 detik
)


In [28]:
import cv2
import mediapipe as mp
import time
from plyer import notification

# Inisialisasi MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# --- PARAMETER DETEKSI & NOTIFIKASI ---
THRESHOLD_ANGLE = 70.0       # Batas mulai bungkuk
ALERT_FRAME_LIMIT = 30       # Frame berturut-turut sebelum dianggap bungkuk
COOLDOWN_SECS = 10          # Jeda waktu antar notifikasi (dalam detik)

# Counter & tracker waktu
slouch_frame_count = 0
last_notif_time = 0  # Menyimpan epoch time saat notifikasi dikirim

cap = cv2.VideoCapture(0)
print("Sistem Notifikasi + Cooldown Aktif! Silakan bungkuk untuk tes.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)
    
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # Ambil koordinat
        nose = landmarks[mp_pose.PoseLandmark.NOSE]
        left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER]
        
        A = (nose.x, nose.y)
        B = (left_shoulder.x, left_shoulder.y)
        C = (right_shoulder.x, right_shoulder.y)
        
        # Hitung sudut di Hidung (A)
        angle = calculate_angle(B, A, C)
        
        # Logika Peringatan
        if angle >= THRESHOLD_ANGLE:
            slouch_frame_count += 1
        else:
            slouch_frame_count = 0
            
        is_slouching = slouch_frame_count >= ALERT_FRAME_LIMIT
        color = (0, 0, 255) if is_slouching else (0, 255, 0)
        
        # --- LOGIKA KIRIM NOTIFIKASI + COOLDOWN ---
        current_time = time.time()
        if is_slouching and (current_time - last_notif_time > COOLDOWN_SECS):
            # Kirim Notifikasi Desktop
            notification.notify(
                title="Postur Buruk Terdeteksi!",
                message="Ayo tegakkan punggungmu, Sabrina!",
                app_name="Stop Bungkuk",
                timeout=3
            )
            last_notif_time = current_time  # Reset waktu notifikasi terakhir
            print("Notifikasi dikirim!")
            
        # --- Visualisasi Segitiga ---
        h, w, _ = frame.shape
        pixel_A = (int(A[0] * w), int(A[1] * h))
        pixel_B = (int(B[0] * w), int(B[1] * h))
        pixel_C = (int(C[0] * w), int(C[1] * h))
        
        cv2.line(frame, pixel_A, pixel_B, color, 2)
        cv2.line(frame, pixel_B, pixel_C, color, 2)
        cv2.line(frame, pixel_C, pixel_A, color, 2)
        
        # Gambar teks sudut di dekat Hidung
        cv2.putText(
            frame, 
            f"Sudut: {angle:.1f} deg", 
            (pixel_A[0] - 60, pixel_A[1] - 20), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            0.6, 
            (255, 255, 255), 
            2
        )
        
    cv2.imshow('Uji Coba Notifikasi', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()


Sistem Notifikasi + Cooldown Aktif! Silakan bungkuk untuk tes.
Notifikasi dikirim!
Notifikasi dikirim!
Notifikasi dikirim!
Notifikasi dikirim!
